# Analýza areálu výskytu vlků 
### vypracovali: Králič Adam, Matějková Barbora

### Zdroje dat
* data výskytu vlků – AOPK, polygonové vrstvy opakovaného možného výskytu vlků a bodové vrstvy způsobených škod a nalezených úhynů
    * https://gis.nature.cz/arcgis/rest/services/DotacniProgramy/OPZP/MapServer
* ArcČR – polygonové vrstvy sídelních jednotek, aproximovaný DMR5G
    * https://www.arcdata.cz/cs-cz/produkty/data/arccr?rsource=https%3A%2F%2Fwww.arcdata.cz%2Fprodukty%2Fgeograficka-data%2Farccr-500
    * https://www.arcgis.com/home/item.html?id=527cdeb48a8b4456a00ed4b35cbed799
* ZABAGED – tématické vrstvy (turistické trasy, VPZCHU)
    * https://geoportal.cuzk.cz/(S(vjgifb34cjzyndrw3xejvw0t))/Default.aspx?mode=TextMeta&text=dSady_zabaged&side=zabaged&menu=24
* Corine Land Cover 2018
    * https://land.copernicus.eu/en/products/corine-land-cover/clc2018

### Analýzy
* porovnání areálu výskytu – s údaji o obyvatelstvu (kolik lidí se nachází v jejich blízkosti), využití krajiny (v jaké krajině jsou nejčastější),
jak moc koreluje jejich výskyt s chráněnými oblastmi nebo jaká je průměrná výška jejich výskytu
* zobrazení dat bodových vrstev – typy poškozené zvěře, trendy v letech
* statistické analýzy
* průnik bodových a polygonových vrstev – bodové údaje vůči sídelním jednotkám

### Použité package
* geopandas
* matlplotlib + patches, lines
* numpy
* branca.element – umožnění legendy pro explore okna přes HTML
* IPython.display (Markdown) – umožnění tvorby markdown buněk z kódu 
* tabulate – tvorba tabulek v Markdown zápisu
* fiona – čtení geopackage souboru
* rasterio – načtení + práce s rastrovými soubory
* rasterstats (zonal_stats) – usnadnění výpočtu statistických informací z rastrového souboru

### Načtení použitých package

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch 
import matplotlib.patches as mpatches 
from matplotlib.lines import Line2D
import numpy as np
from branca.element import Template, MacroElement
from IPython.display import Markdown as md # tento package umožňuje vytvořit markdown buňku přímo v python bloku
# Zdroj: https://data-dive.com/jupyterlab-markdown-cells-include-variables/
from tabulate import tabulate
import fiona
from rasterstats import zonal_stats
import rasterio
from rasterio.mask import mask


### Areál výskytu vlků
* načtení primárních dat – výskyt po červnu 2024 (čtení zip souboru přes gpd), výskyt před červnem 2024 (čtení shp souboru přes gpd)
* načtení sekundárních dat – hranice ČR (čtení shp souboru přes gpd)
* vykreslení – všechny vrstvy v jednom plot okně (vrchní vrstva 50% průhledností), přes matplotlib
* vykreslení legendy – pomocí matplotlib.patches, matplotlib.lines (volba geometrie, barvy a názvu prvku v legendě)

In [ ]:
Zipfile_vlci_path = "data/vlk_vyskyt.zip"
nested_file_vlci_path = "vlk_vyskyt.shp"

vyskyt_vlka_po_24 = gpd.read_file(f"{Zipfile_vlci_path}!{nested_file_vlci_path}")
file_vlci_vyskyt_pred_24 = "data/vyskyt_vlku_pred_0724/vyskyt_vlku_pred_25.shp"
vyskyt_vlka_pred_24 = gpd.read_file(file_vlci_vyskyt_pred_24)

hranice_file = "data/cr/hranice.shp"
hranice = gpd.read_file(hranice_file)

In [ ]:
# vykreslení vrstev
ax = vyskyt_vlka_pred_24.plot(color="red", alpha=1)
vyskyt_vlka_po_24.plot(ax=ax, color="blue", alpha=0.5)
hranice.plot(ax=ax, facecolor="none", edgecolor="black")

# vytvoření vlastní legendy
legend_elements = [
    Patch(facecolor="red", edgecolor="r", label="před červencem 2024"),
    Patch(facecolor="blue", edgecolor="b", alpha=0.5, label="po červenci 2024"),
]

ax.legend(handles=legend_elements, loc="upper right")
plt.show()


## Celkový počet způsobených škod
* načtení bodové vrstvy ve formátu shp přes geopandas
* statistika celkového počtu napadených zvířat
* vykreslení bodové vrstvy pomocí gdf.explore a vytvoření legendy přes HTML příkaz (Branca.Element)

In [ ]:
file_poskozeni_vlci = "data/skody_od_vlku/skody_zpusobene_vlky.shp"
poskozeni_vlky = gpd.read_file(file_poskozeni_vlci)
statistika_poskozeni = poskozeni_vlky[["OVCE_MRT", "OVCE_ZRAN", "KOZY_MRT", "KOZY_ZRAN", "SKOT_MRT", "SKOT_ZRAN", "KUN_MR_ZR"]].sum(numeric_only=True)

# vytvoření markdown příkazu s tabulkou
md_table = tabulate(gpd.GeoDataFrame(statistika_poskozeni), headers=["Typ zvěře a poškození", "počet celkových napadených zvířat"], tablefmt='github', showindex=True)

# volání markdown metody
md("### Napadená zvířata\n" + md_table)

In [ ]:
m = poskozeni_vlky.explore(legend=False)

# volba barev v pořadí proměnné cols
colors = [
    "#E41A1C",  # red
    "#377EB8",  # blue
    "#4DAF4A",  # green
    "#984EA3",  # purple
    "#FF7F00",  # orange
    "#FFFF33",  # yellow
    "#A65628"   # brown
]

# volba vykreslených atributů
cols = [
    "OVCE_MRT", "OVCE_ZRAN", "KOZY_MRT", "KOZY_ZRAN",
    "SKOT_MRT", "SKOT_ZRAN", "KUN_MR_ZR"
]

# vykreslení bod s příslušným atributem
for col, color in zip(cols, colors):
    poskozeni_vlky[poskozeni_vlky[col] > 0].explore(
        m=m,
        color=color,
        marker_kwds={
        "radius": 5,  
        "opacity": 1
        },
        legend=True,
        tooltip=["OBEC", "DATUM_SKOD", "OVCE_MRT", "OVCE_ZRAN", "KOZY_MRT", "KOZY_ZRAN", "SKOT_MRT", "SKOT_ZRAN", "KUN_MR_ZR"],
        name=col
    )

# HTML příkaz pro tvorbu legendy
legend_html = """
{% macro html(this, kwargs) %}
<div style="
    position: fixed;
    bottom: 40px;
    left: 40px;
    z-index: 9999;
    background-color: white;
    padding: 10px;
    border: 2px solid grey;
    border-radius: 5px;
">
<b>napadená zvířata</b><br>
&nbsp;<i style="color:#E41A1C">●</i>&nbsp; mrtvé ovce<br> 
&nbsp;<i style="color:#377EB8">●</i>&nbsp; zraněné ovce<br>
&nbsp;<i style="color:#4DAF4A">●</i>&nbsp; mrtvé kozy<br>
&nbsp;<i style="color:#984EA3">●</i>&nbsp; zraněné kozy<br>
&nbsp;<i style="color:#FF7F00">●</i>&nbsp; mrtvý skot<br>
&nbsp;<i style="color:#FFFF33">●</i>&nbsp; zraněný skot<br>
&nbsp;<i style="color:#A65628">●</i>&nbsp; mrtví/zranění koně
</div>
{% endmacro %}
"""

macro = MacroElement()
macro._template = Template(legend_html)
m.get_root().add_child(macro) # přidání legendy do mapy

m


### Analýza úhynu vlků
* načtení dat
* vykreslení dle typu příčiny úhynu
* vypsání úhynu dle roků (2012 – 2025)

In [ ]:
file_uhyn_vlku = "data/uhyn_vlku/uhyn_vlku.shp" # nahrání souboru úhynu vlků
uhyn_vlku = gpd.read_file(file_uhyn_vlku)

In [ ]:
# definování barev
colors = ["red", "blue", "green", "orange"]  

# vykreslení podle typu příčiny úhynu pomocí explore
uhyn_vlku.explore(
    column="PRICINA",
    categorical=True,
    legend=True,
    legend_kwds={"caption": "Příčina úhynu vlků"},
    cmap=colors,
    marker_kwds={"radius": 6, "fillOpacity": 0.8}
)


In [ ]:
uhyn_s_roky = uhyn_vlku["ROK"].value_counts()   # získání počtu jednotlivých hodnot atributů
uhyn_s_roky_sorted = uhyn_s_roky.sort_index(ascending=False)  # seřazeno dle roku

md_text = (
    "### Počet uhynulých vlků dle roků\n\n"
    "| rok | počet uhynulých vlků |\n"
    "|-----|--------------------|\n"
    + "\n".join(f"| {year} | {count} |" for year, count in uhyn_s_roky_sorted.items())
)

md(md_text)


### Analýza nejzasaženějších obcí
* načtení polygonové vrstvy obcí ve formátu shp pomocí geopandas
* spojení vrstvy obcí s bodovou vrstvou poškození od vlků pomocí prostorového příkazu gpd.sjoin()
* vykreslení poškozených obcí
* vypsání nejpoškozenějších obcí
* vypsání celkového počtu obyvatel v poškozených obcích (kolika lidí se to týká)

In [ ]:
# načtení polygonové vrstvy obcí
file_obce = "data/obce/obce.shp"
obce = gpd.read_file(file_obce)

# prostorové propojení vrstvy obcí a bodové vrstvy poškození vlky
obce_s_vlky = obce.join(gpd.sjoin(poskozeni_vlky, obce).groupby("KOD_OBEC").size().rename("zvirata_sezrana_vlky"),
    on="KOD_OBEC",
    how="left"
)

In [ ]:
# vyfiltrování obcí, kde došlo k poškození
obce_s_vlky = obce_s_vlky[obce_s_vlky["zvirata_sezrana_vlky"] > 0]

In [ ]:
# vykreslení poškozených obcí
obce_s_vlky.explore(column='zvirata_sezrana_vlky', legend=True, cmap='OrRd', popup=["NAZ_OBEC", "zvirata_sezrana_vlky"], tooltip=False) # zobrazuje obce dle poskozeni


In [ ]:
# srovnání a limit dat
pocet_obci = 10
obce_s_vlky = obce_s_vlky.sort_values("zvirata_sezrana_vlky", ascending=False)
obce_s_vlky_filt = obce_s_vlky[["NAZ_OBEC", "zvirata_sezrana_vlky"]].head(pocet_obci) # nejpoškozenější

# vytvoření markdown příkazu s tabulkou
md_table = tabulate(obce_s_vlky_filt, headers=["název obce", "počet napadených zvířat"], tablefmt='github', showindex=False)

# volání markdown metody
md("### Nejzasaženější obce\n" + md_table)


In [ ]:
md(f"### Počet obyvatel v lokalitách, kde vlci působí škody: {int(obce_s_vlky["POCET_OBYV"].sum())}")

## Poškození vlky VS výskyt vlků
* vytvoření pomocných funkcí
    * def *plot_vyskyt_vlku()* – vykreslení výskytu vlků za dané období
    * def *vlci_mimo_vyskyt()* – využití prostorové operace sjoin a nalezení prvků, které nebyly spojeny 
* analýza pro oba typy výskytu (před červencem 2024 a po $\rightarrow$ průnik těchto výsledků)
* vypsání obcí s potvrzenými výskyty vlků mimo areál jejich výskytu
* vykreslení obou typů výskytu a bodové vrstvy s poškozeními
* vypsání počtu obyvatel, které mají právo na náhradu škody způsobenou vlkem

In [ ]:
def plot_vyskyt_vlku(hranice, poskozeni, vyskyt, roky):
    # vykreslení výskytu vlků za dané období
    figsize=(12,10)
    
    # vykreslení mapy
    ax = vyskyt.plot(color="#639472", figsize=figsize)
    bx = poskozeni[poskozeni["ROK"].isin(roky)].plot(ax=ax, color="#B82B2B", markersize=2)
    hranice.plot(ax=bx, facecolor="None", label="hranice")    
    
    # vytvoření vlastní legendy přes matplotlib.[patches, lines]
    legend_elements = [
        Patch(facecolor="#639472", edgecolor="black", alpha=0.5, label="výskyt vlků"),
        Line2D([0], [0], marker='o', color='w', label="poškození vlky",
               markerfacecolor="#B82B2B", markersize=10)
    ]
    
    # přidání legendy do mapy
    ax.legend(handles=legend_elements, loc="upper right")
    
def vlci_mimo_vyskyt(obce, vyskyt, poskozeni):
    # nalezení obcí, kde došlo k poškození, ale nenachází se v areálu výskytu
    joined = gpd.sjoin(poskozeni, vyskyt, predicate="within", how="left")
    
    # vybrání bodů poškození, které nejsou spojené
    poskozeni_mimo_vyskyt = joined[joined["index_right"].isna()]
    poskozeni_union = poskozeni_mimo_vyskyt.union_all()
    
    # nalezení obcí, ve kterých došlo k vybraným poškozením
    obce_s_vlky = obce[obce.intersects(poskozeni_union)]
    return obce_s_vlky
    

In [ ]:
obce_s_vlky_pred_24 = vlci_mimo_vyskyt(obce, vyskyt_vlka_pred_24, poskozeni_vlky[poskozeni_vlky["ROK"] < 2024])
obce_s_vlky_po_24 = vlci_mimo_vyskyt(obce, vyskyt_vlka_po_24, poskozeni_vlky[poskozeni_vlky["ROK"] >= 2024])
obce_s_vlky = gpd.pd.concat([obce_s_vlky_pred_24, obce_s_vlky_po_24]).drop_duplicates("KOD_OBEC") # obce mimo výskyt vlků, kde proběhlo k napadení zvěře

# vytvoření markdown příkazu
items = obce_s_vlky["NAZ_OBEC"].tolist() # přepínání vrstev
md_text = "### Obce s potvrzenými výskyty vlků mimo areál jejich výskytu\n" + "\n".join(f"- {item}" for item in items)

# zobrazení markdown buňky
md(md_text)


In [ ]:
obce_s_vlky.explore(tooltip=False, popup=["NAZ_OBEC", 'POCET_OBYV'], color="red") # vykreslení obcí, kde došlo k poškození mimo areál výskytu

In [ ]:
# vykreslení výskytu a poškození vlků před rokem 2024
plot_vyskyt_vlku(hranice, poskozeni_vlky, vyskyt_vlka_pred_24, [2018, 2019, 2020, 2021, 2022, 2023])

In [ ]:
# vykreslení výskytu vlků po roce 2024
plot_vyskyt_vlku(hranice, poskozeni_vlky, vyskyt_vlka_po_24, [2024, 2025])

In [ ]:
md(f"### Počet obyvatel v obcích, kde vlci působí škody, ale nemají právo na odškodnění: {int(obce_s_vlky["POCET_OBYV"].sum())}")

## Snězená zvěř dle okresů
* vytvořená funkce - def *zobraz_poskozeni_v_okresech()*
    * využití prostorového spojení bodové vrstvy poškození a polygonové vrstvy okresů metodou sjoin
    * agregace atributů dle okresů (součet hodnot atributů, které obsahují informace o počtu napadené zvěře)
    * vykreslení metodou explore()
* vykreslení pouze těch okresů, ve kterých bylo alespoň 1 poškozené zvíře

In [ ]:
okresy = gpd.read_file("data/okresy/okresy.shp") # načtení souboru

In [ ]:
def zobraz_poskozeni_v_okresech(okresy, poskozeni, zobrazeny_atribut):
    okresy_s_poskozenim = (  # provedení spation join operace
        okresy.sjoin(poskozeni)
            .groupby("KOD_OKRES")  # group by okres
            .agg({
                "NAZ_LAU1": "first",
                "KOD_OKRES": "first",
                "geometry": "first",
                "OVCE_MRT": "sum",
                "OVCE_ZRAN": "sum",
                "KOZY_MRT": "sum",
                "KOZY_ZRAN": "sum",
                "SKOT_MRT": "sum",
                "SKOT_ZRAN": "sum",
                "KUN_MR_ZR": "sum"
            })
            .sort_values(zobrazeny_atribut, ascending=False)
    )
    okresy = okresy.merge(
        okresy_s_poskozenim[["NAZ_LAU1", zobrazeny_atribut]],
        on="NAZ_LAU1",
        how="left"
    )
    
    return okresy[["NAZ_LAU1",zobrazeny_atribut, "geometry"]].sort_values(zobrazeny_atribut, ascending=False)


In [ ]:
atribut = "KOZY_MRT" # které zvíře a stav poškození, alternativně: KOZY_ZRAN, SKOT_MRT, SKOT_ZRAN, KUN_MR_ZR, OVCE_ZRAN, OVCE_MRT
okresy_s_atributem = zobraz_poskozeni_v_okresech(okresy,poskozeni_vlky, atribut)

okresy_s_atributem.assign(
    atribut_plot=lambda df: df[atribut].replace(0, np.nan)
).explore(
    column="atribut_plot",
    cmap="YlOrRd",
    legend_kwds={"caption": "počet " + atribut},
    legend=True,
    tooltip=["NAZ_LAU1", atribut],  # popup s pouze těmito attributy
    missing_kwds={
        "color": "lightgray",
        "label": "0"
    }
)

## Analýza typu krajiny (CLC) v areálu výskytu vlků
* načtení geodatabáze pomocí package Fiona (gdb obsahuje sídelní jednotky a corine land cover 2018)
* převedení CLC do crs 5514
* prostorová operace clip podle výskytu vlků po roce červenci 2024 (doba trvání: 24 minut)
* $\rightarrow$ tvorba oříznutého CLC a zpětné načtení pro úsporu času
* rozdělení CLC podle kódu krajiny do jednotlivých krajinných typů
* statistika počtu polygonů jednotlivých typů
* rozloha jednotlivých typů krajiny v Ha
* vykreslení oříznuté CLC přes matplotlib.pyplot 

In [ ]:
# načtení geodatabáze
# zdroj: https://gis.stackexchange.com/questions/479431/open-gdb-file-in-python

file_geodatabase = r"data/ZSJ_CLC.gdb" #A file geodatabase with multiple vector layers in it

# vypsání jednotlivých souborů v geodatabázi
for feature_class in fiona.listlayers(file_geodatabase):
    print(feature_class)
    
clc = gpd.read_file(filename=file_geodatabase, layer="CORINE_Land_Cover_2018_CZ") # uloží pouze požadovanou vrstvu

In [ ]:
clc5514 = clc.to_crs(5514) # převedení Corine Land Cover do Křováka pro umožnění prostorových operací s vrstvou výskytu vlků

In [ ]:
#clc_vlci = gpd.clip(clc5514, vyskyt_vlka_po_24) # dlouhé, nepouštět při prezentaci!

In [ ]:
#clc_vlci.to_file('data/CLC_vlci/clc_vlci.shp')
#clc_vlci.plot()

In [ ]:
clc_vlci_shp = gpd.read_file('data/CLC_vlci/clc_vlci.shp')

In [ ]:
# podmínky pro rozdělení krajiny dle kódů
conditions = [   
    clc_vlci_shp["Code_18"].isin(['111', '112', '121', '122', '123', '124']),
    clc_vlci_shp["Code_18"].isin(['131', '132', '133']),
    clc_vlci_shp["Code_18"].isin(['141', '142']),
    clc_vlci_shp["Code_18"].isin([str(x) for x in range(211, 244)]),
    clc_vlci_shp["Code_18"].isin([str(x) for x in range(311, 325)]),
    clc_vlci_shp["Code_18"].isin(['332', '333']),
    clc_vlci_shp["Code_18"].isin(['411', '412']),
    clc_vlci_shp["Code_18"].isin(['511', '512'])
]

# názvy jednotlivých typů krajiny
choices = ["zástavba",
           "lomy či neplodná půda",
           "zástavba se zelení",
           "zemědělská půda",
           "lesní plochy",
           "plochy bez/s řídkou vegetací",
           "močály a bažiny",
           "vodní plochy"
           ]

# výpočet patříčného typu funkcí select
clc_vlci_shp["typ"] = np.select(conditions, choices, default="000")

In [ ]:
# rozložení polygonů dle typu krajiny
clc_vlci_dissolve = clc_vlci_shp.dissolve("typ", 
        aggfunc = {
        "Code_18": lambda x: ", ".join(sorted(x.unique())),
        "Remark": "first",
        "Area_Ha": "sum",
        "ID": "first",
        "Shape_Leng": "sum",
        "Shape_Area": "sum"
    })

In [ ]:
# seřazení CLC typů dle výměry v Ha
clc_rozloha = sorted(clc_vlci_dissolve["Area_Ha"].items(),
    key=lambda x: -x[1],  # seřadí podle Ha
)

# vytvoření markdown příkazu
md_text = (
    "### Rozloha typů ploch z Corine Land Cover v areálu výskytu vlků\n\n"
    "| Typ | Rozloha [ha] |\n"
    "|-----|-------|\n"
    + "\n".join(f"| {typ} | {round(rozloha,2)} |" for typ, rozloha in clc_rozloha)
)

# volání markdown příkazu
md(md_text)

In [ ]:
# volba barev pro jednotlivé typy krajiny
color_dict = {
    "zástavba": "#e41a1c",                
    "lomy či neplodná půda": "#aab4a9",  
    "zástavba se zelení": "#b8695e",    
    "zemědělská půda": "#cbf074",       
    "lesní plochy": "#3B8B34",         
    "plochy bez/s řídkou vegetací": "#cee7ba", 
    "močály a bažiny": "#2a7e7e",       
    "vodní plochy": "#3072D6"           
}

# reset indexu (aby nebyl "typ" nadále indexem a mohlo to být použito jako atribut)
clc_vlci_dissolve = clc_vlci_dissolve.reset_index()

# vytvoření nového atributu barvy
clc_vlci_dissolve["color"] = clc_vlci_dissolve["typ"].map(color_dict)

# vykreslení krajiny z CLC v areálu výskytu vlků
ax = clc_vlci_dissolve.plot(color=clc_vlci_dissolve["color"], edgecolor="None", linewidth=0.3, figsize=(19, 15))
handles = [mpatches.Patch(color=color, label=typ) for typ, color in color_dict.items()] # vykreslení legendy
ax.legend(handles=handles, title="typ krajiny")
ax.set_axis_off()
hranice.plot(ax=ax, facecolor="none", edgecolor="black")
plt.show()

## Turistické trasy s možným výskytem vlků
* načtení turistických tras ve formátu shp ze ZABAGED
* výpočet celkové délky tras
* oříznutí tras metodou overlay podle areálu výskytu vlků po červenci 2024 (doba trvání: 15 minut)
* $\rightarrow$ tvorba oříznutých tur. tras a zpětné načtení pro úsporu času
* výpočet délky tras v areálu výskytu vlků
* vykreslení tras přes matplotlib.pyplot 
* poměr tras s možným výskytem vlků vůči celkové delkce tras 

In [ ]:
# načtení souboru
tur_trasy = gpd.read_file("data/turisticke_trasy/turisticke_trasy.shp")
total_length = tur_trasy.length.sum()
#vlci_trasy = gpd.overlay(tur_trasy, vyskyt_vlka_po_24, how='intersection') # Intersekce výskytu vlka s turistickými trasami, dlouhé, nepouštět při prezentaci!

In [ ]:
#vlci_trasy.to_file("data/turisticke_trasy/vlci_trasy.shp") # uložení do souboru

In [ ]:
# načtení oříznutých tras
vlci_trasy = gpd.read_file("data/turisticke_trasy/vlci_trasy.shp")

In [ ]:
figsize=(12,10)

# vykreslení mapy
ax = vyskyt_vlka_po_24.plot(color="#B7D5C0", figsize=figsize)
bx = tur_trasy.plot(ax=ax, color="#B18867", markersize=1)
cx = vlci_trasy.plot(ax=bx, color="#DA5635", markersize=1)
hranice.plot(ax=cx, facecolor="None", label="hranice")    

# vytvoření vlastní legendy
legend_elements = [
    Patch(facecolor="#B7D5C0", edgecolor="black", label="výskyt vlků"),
    Line2D([0], [0], color="#B18867", linewidth=2, label="turistické trasy"),
    Line2D([0], [0], color="#DA5635", linewidth=2, label="trasy ve výskytu vlků"),
]

# přidání legendy do mapy
ax.legend(handles=legend_elements, loc="upper right")

In [ ]:
# výpočet délky tras v areálu výskytu vlků
vlci_trasy_length = vlci_trasy.length.sum()
procento_tras = vlci_trasy_length / total_length * 100
procento_tras

md(f"### **{round(procento_tras, 1)} %** turistických tras se nachází v areálu výskytu vlků")

## Výskyt vlků v chráněných územích
* načtení shp souboru velkoplošných zvláště chráněných území ze ZABAGED pomocí geopandas
* vytvořen průnik VZCHU s areálem výskytu vlků pomocí metody *intersection()*
* výpočet celkové rozlohy obou souborů $\rightarrow$ výpočet VZCHU v areálu výskytu vlků v %
* vykreslení pomocí matplotlib.pyplot

In [ ]:
# načtení souboru
chranene_uzemi = gpd.read_file("data/VPZCHU/VPZ_chranene_uzemi.shp") 

In [ ]:
# výpočetní proces
chranene_uzemi_union = chranene_uzemi.union_all()   # sjednocení obou vrstev
vyskyt_vlka_union = vyskyt_vlka_po_24.union_all()
prunik = chranene_uzemi_union.intersection(vyskyt_vlka_union) # vzájemný průnik
plocha_chraneneho_uzemi = chranene_uzemi_union.area    
plocha_pruniku = prunik.area

# výpočet plochy průniku v procentech
plocha_v_procentech = (plocha_pruniku / plocha_chraneneho_uzemi) * 100

md(f"### **{round(plocha_v_procentech, 1)} %** plochy chráněných území může obsahovat vlky")


In [ ]:
# vykreslení ploch přes sebe
ax = vyskyt_vlka_po_24.plot(color="gray", alpha=0.5, legend=True, figsize=[12,10]) # nevypisuje legendu?
bx = chranene_uzemi.plot(ax = ax, color="green", alpha=0.5, legend=True)
hranice.plot(ax=bx, facecolor="None", label="hranice")

# vytvoření vlastní legendy
legend_elements = [
    Patch(facecolor="gray", edgecolor="gray", alpha=0.5, label="výskyt vlků po červenci 2024"),
    Patch(facecolor="green", edgecolor="green", alpha=0.5, label="velkoplošné chráněné území"),
]

# přidání legendy
ax.legend(handles=legend_elements, loc="upper right")



## Průměrná nadmořská výška výskytu vlka
* načtení DMR ve formátu tif přes rasterio package 
* výpočet statistik v areálu výskytu vlků pomocí rasterstats package
* výpočet průměrné nadmořské výšky 
* vykreslení DMR v areálu výskytu vlků (použití masky pro oříznutí dat, změna nodata na hodnotu 0)

In [ ]:
# načtení rastrového souboru

dmr_file = "data/dmr/dmr.tif"
dmr = rasterio.open(dmr_file)

In [ ]:
# nastavení statistických parametrů
stats = zonal_stats(
    vectors=vyskyt_vlka_po_24,
    raster=dmr_file,
    stats=["mean", "max"],          
    geojson_out=True
)

# výpočet rastrových statistik
result = gpd.GeoDataFrame.from_features(stats)

# získá atribut průměrné hodnoty
prumerna_vyska = result["mean"].iloc[0]

# vypsání hodnoty
md(f"### Průměrná výška výskytu vlka: {round(prumerna_vyska, 3)} m")

In [ ]:
# vyextrahování geometrie
poly = vyskyt_vlka_po_24.iloc[0].geometry # vezme první (jediný) polygon
geom = [poly]

# maskovaný rastr
clipped, _ = mask(dmr, geom, crop=True)
data = clipped[0].astype(float)

data[data == 32767] = 0 # vyplnění nodata za vhodnější konstantu

# Plot
plt.figure(figsize=(13, 10))
plt.imshow(data, cmap="terrain")
plt.colorbar(label="Nadmořská výška [m]")
plt.title("Výškový výskyt vlků")
plt.show()